In [0]:
import pandas as pd

feat = spark.sql("""
    SELECT event_date, agent_name,
           COUNT(*) AS calls,
           AVG(prompt_tokens) AS avg_prompt_tokens,
           CAST(SUM(cost_usd) / COUNT(*) AS DOUBLE) AS cost_per_call,
           AVG(latency_ms) AS avg_latency_ms,
           AVG(CASE WHEN status <> 'success' THEN 1.0 ELSE 0.0 END) AS error_rate
    FROM dbw_agentops.gold.fact_agent_calls
    WHERE event_date < current_date()
    GROUP BY event_date, agent_name
""").toPandas().sort_values(["agent_name", "event_date"]).reset_index(drop=True)

print(len(feat), "agent-day rows")
display(feat[feat.agent_name == "support-bot"])

In [0]:
import numpy as np

features = ["avg_prompt_tokens", "cost_per_call", "avg_latency_ms", "error_rate"]

def robust_z(s):
    med = s.median()
    mad = (s - med).abs().median()
    return 0.6745 * (s - med) / mad if mad > 0 else s * 0

for f in features:
    feat[f"z_{f}"] = feat.groupby("agent_name")[f].transform(lambda s: robust_z(s.astype(float)))

zcols = [f"z_{f}" for f in features]
feat["max_abs_z"] = feat[zcols].abs().max(axis=1)
feat["stat_anomaly"] = feat["max_abs_z"] > 3.5

print(feat["stat_anomaly"].sum(), "flagged of", len(feat))
display(feat[feat["stat_anomaly"]].sort_values("max_abs_z", ascending=False)
        [["event_date", "agent_name", "avg_prompt_tokens", "cost_per_call", "avg_latency_ms", "error_rate", "max_abs_z"]])

In [0]:
import mlflow, mlflow.sklearn
from mlflow.models import infer_signature
from sklearn.ensemble import IsolationForest

mlflow.set_registry_uri("databricks-uc")

# Ground truth: we planted the spike on support-bot, Oct 24
feat["is_planted_spike"] = (feat.agent_name == "support-bot") & (feat.event_date.astype(str) == "2026-10-24")

def evaluate(flag):
    tp = int((flag & feat.is_planted_spike).sum())
    fp = int((flag & ~feat.is_planted_spike).sum())
    fn = int((~flag & feat.is_planted_spike).sum())
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    return precision, recall

X = feat[zcols]

# Run 1: the statistical baseline
with mlflow.start_run(run_name="robust_zscore_baseline"):
    mlflow.log_param("method", "robust_zscore")
    mlflow.log_param("threshold", 3.5)
    p, r = evaluate(feat["stat_anomaly"])
    mlflow.log_metric("precision", p)
    mlflow.log_metric("recall", r)
    mlflow.log_metric("n_flagged", int(feat["stat_anomaly"].sum()))
    print(f"baseline:         precision={p:.2f} recall={r:.2f}")

# Run 2: Isolation Forest
with mlflow.start_run(run_name="isolation_forest"):
    params = {"n_estimators": 200, "contamination": 0.04, "random_state": 42}
    model = IsolationForest(**params).fit(X)
    feat["if_score"] = -model.score_samples(X)          # higher = more anomalous
    feat["if_anomaly"] = model.predict(X) == -1
    p, r = evaluate(feat["if_anomaly"])
    mlflow.log_param("method", "isolation_forest")
    mlflow.log_params(params)
    mlflow.log_metric("precision", p)
    mlflow.log_metric("recall", r)
    mlflow.log_metric("n_flagged", int(feat["if_anomaly"].sum()))
    mlflow.sklearn.log_model(
        model, artifact_path="model",
        signature=infer_signature(X, model.predict(X)),
        registered_model_name="dbw_agentops.gold.agent_anomaly_iforest")
    print(f"isolation forest: precision={p:.2f} recall={r:.2f}")

display(feat.sort_values("if_score", ascending=False).head(5)
        [["event_date", "agent_name", "avg_prompt_tokens", "cost_per_call", "if_score", "if_anomaly"]])

In [0]:
out = feat[["event_date", "agent_name", "calls", "avg_prompt_tokens", "cost_per_call",
            "avg_latency_ms", "error_rate", "max_abs_z", "stat_anomaly", "if_score", "if_anomaly"]]
(spark.createDataFrame(out)
      .write.mode("overwrite").option("overwriteSchema", "true")
      .saveAsTable("dbw_agentops.gold.agent_anomalies"))

print(spark.table("dbw_agentops.gold.agent_anomalies").count(), "rows saved")
display(spark.table("dbw_agentops.gold.agent_anomalies")
        .filter("stat_anomaly OR if_anomaly").orderBy("if_score", ascending=False))